# Assignment 2 — Robo-Football

A complete turn-based grid game, rule checks, heuristic agents, minimax, alpha-beta, MCTS rollouts, and experiments.

## Rules and explicit design decisions

The board has 6 rows and 5 columns. Team A attacks row 5 and Team B attacks row 0.
Each side has four outfield players and one goalkeeper. Teams alternate turns strictly.
The goalkeeper moves sideways on its own goal line only. A dribble moves one of eight directions
and is tackled if an opponent is adjacent to the destination. A pass travels along a straight
horizontal, vertical, or exact diagonal line and is blocked only by an opponent on that line.
A shot is permitted in the attacking two rows; it scores unless the opposing goalkeeper is within
one column, in which case the goalkeeper saves and gains possession.

Design decisions: player positions are stored as tuples; the goalkeeper is index 4; a successful
shot resets the formation and gives the conceding team possession; a save gives the goalkeeper's
team possession. When multiple tacklers are adjacent, the lowest player index wins. The match
budget is a fixed number of plies (individual team turns).

In [ ]:
from dataclasses import dataclass
from collections import defaultdict
import random
import time
import math

ROWS, COLS = 6, 5
A, B = 0, 1

def initial_positions():
    a = ((1, 1), (1, 3), (2, 0), (2, 4), (0, 2))
    b = tuple((ROWS - 1 - r, c) for r, c in a)
    return (a, b)

@dataclass(frozen=True)
class GameState:
    positions: tuple
    possession: int
    carrier: int
    goals: tuple
    turn: int
    turns_left: int

def initial_state(turn_budget=80):
    return GameState(initial_positions(), A, 0, (0, 0), A, turn_budget)

def inside(pos):
    r, c = pos
    return 0 <= r < ROWS and 0 <= c < COLS

def occupied(state):
    return {pos for team in state.positions for pos in team}

def adjacent(p, q):
    return max(abs(p[0] - q[0]), abs(p[1] - q[1])) == 1

def line_cells(p, q):
    dr, dc = q[0] - p[0], q[1] - p[1]
    if dr == 0 and dc != 0:
        step = (0, 1 if dc > 0 else -1)
    elif dc == 0 and dr != 0:
        step = (1 if dr > 0 else -1, 0)
    elif abs(dr) == abs(dc) and dr != 0:
        step = (1 if dr > 0 else -1, 1 if dc > 0 else -1)
    else:
        return []
    cells = []
    r, c = p[0] + step[0], p[1] + step[1]
    while (r, c) != q:
        cells.append((r, c))
        r += step[0]
        c += step[1]
    return cells

def attacking_third(team, pos):
    return pos[0] >= 4 if team == A else pos[0] <= 1

def legal_actions(state):
    team = state.turn
    own = state.positions[team]
    other = state.positions[1 - team]
    taken = occupied(state)
    actions = []
    if state.possession == team:
        carrier_pos = own[state.carrier]
        for dr in (-1, 0, 1):
            for dc in (-1, 0, 1):
                if dr == 0 and dc == 0:
                    continue
                dest = (carrier_pos[0] + dr, carrier_pos[1] + dc)
                if inside(dest) and dest not in taken:
                    actions.append(("dribble", dest))
        for teammate in range(5):
            if teammate == state.carrier:
                continue
            target = own[teammate]
            cells = line_cells(carrier_pos, target)
            if cells and not any(p in other for p in cells):
                actions.append(("pass", teammate))
        if attacking_third(team, carrier_pos):
            actions.append(("shoot",))
    else:
        for player, pos in enumerate(own):
            if player == 4:
                candidates = [(pos[0], pos[1] + dc) for dc in (-1, 1)]
            else:
                candidates = [(pos[0] + dr, pos[1] + dc)
                              for dr in (-1, 0, 1) for dc in (-1, 0, 1)
                              if (dr, dc) != (0, 0)]
            for dest in candidates:
                if inside(dest) and dest not in taken:
                    if player == 4 and dest[0] != (0 if team == A else ROWS - 1):
                        continue
                    actions.append(("defend", player, dest))
    return actions

def reset_after_goal(state, scoring_team):
    conceded = 1 - scoring_team
    return GameState(initial_positions(), conceded, 0,
                     (state.goals[0] + (scoring_team == A),
                      state.goals[1] + (scoring_team == B)),
                     conceded, state.turns_left - 1)

def apply_action(state, action):
    team = state.turn
    positions = [list(state.positions[0]), list(state.positions[1])]
    possession, carrier = state.possession, state.carrier
    if action[0] == "dribble":
        positions[team][carrier] = action[1]
        tacklers = [i for i, p in enumerate(positions[1-team]) if adjacent(p, action[1])]
        if tacklers:
            possession = 1 - team
            carrier = tacklers[0]
    elif action[0] == "pass":
        carrier = action[1]
    elif action[0] == "shoot":
        keeper_col = positions[1-team][4][1]
        shooter_col = positions[team][carrier][1]
        if abs(keeper_col - shooter_col) <= 1:
            possession, carrier = 1-team, 4
        else:
            new_state = GameState(tuple(tuple(x) for x in positions), possession, carrier,
                                  state.goals, 1-team, state.turns_left)
            return reset_after_goal(new_state, team)
    elif action[0] == "defend":
        _, player, dest = action
        positions[team][player] = dest
    return GameState(tuple(tuple(x) for x in positions), possession, carrier,
                     state.goals, 1-team, state.turns_left - 1)

def terminal(state):
    return state.turns_left <= 0

def render(state):
    board = [["." for _ in range(COLS)] for _ in range(ROWS)]
    for team in (A, B):
        for i, (r, c) in enumerate(state.positions[team]):
            board[r][c] = ("A" if team == A else "B") + ("K" if i == 4 else str(i))
    for r in range(ROWS):
        print(" ".join(board[r]))
    print("Possession:", "A" if state.possession == A else "B",
          "carrier:", state.carrier, "score:", state.goals,
          "next:", "A" if state.turn == A else "B",
          "turns left:", state.turns_left)

## Automated rule checks

In [ ]:
def run_rule_checks():
    s = initial_state()
    assert len(occupied(s)) == 10
    assert len(legal_actions(s)) > 0
    assert all(action[0] in {"dribble", "pass", "shoot", "defend"} for action in legal_actions(s))
    for action in legal_actions(s):
        ns = apply_action(s, action)
        assert len(occupied(ns)) == 10
        assert all(inside(p) for p in occupied(ns))
        assert ns.turn != s.turn
    assert attacking_third(A, (4, 2))
    assert attacking_third(B, (1, 2))
    assert not attacking_third(A, (3, 2))
    assert line_cells((0, 0), (3, 3)) == [(1, 1), (2, 2)]
    assert line_cells((0, 0), (2, 1)) == []

    tackle_positions = (
        ((2, 2), (1, 0), (1, 4), (2, 0), (0, 2)),
        ((3, 3), (4, 0), (4, 2), (4, 4), (5, 2))
    )
    tackle_state = GameState(tackle_positions, A, 0, (0, 0), A, 10)
    tackle_action = ("dribble", (2, 3))
    tackled = apply_action(tackle_state, tackle_action)
    assert tackled.possession == B and tackled.carrier == 0

    save_positions = (
        ((4, 1), (1, 1), (2, 0), (2, 4), (0, 2)),
        ((3, 0), (3, 1), (3, 3), (4, 4), (5, 2))
    )
    save_state = GameState(save_positions, A, 0, (0, 0), A, 10)
    assert ("shoot",) in legal_actions(save_state)
    saved = apply_action(save_state, ("shoot",))
    assert saved.possession == B and saved.carrier == 4
    assert saved.goals == (0, 0)

    goal_positions = (
        ((4, 1), (1, 1), (2, 0), (2, 4), (0, 2)),
        ((3, 0), (3, 1), (3, 3), (4, 4), (5, 4))
    )
    goal_state = GameState(goal_positions, A, 0, (0, 0), A, 10)
    scored = apply_action(goal_state, ("shoot",))
    assert scored.goals == (1, 0)
    assert scored.possession == B and scored.turn == B
    assert scored.positions == initial_positions()

    non_attacking = GameState(initial_positions(), A, 0, (0, 0), A, 10)
    assert ("shoot",) not in legal_actions(non_attacking)
    return "All basic, tackle, save, goal, and illegal-shot checks passed."

run_rule_checks()

## Evaluation functions and baseline agents

In [ ]:
def evaluate_field(state, team):
    opponent = 1 - team
    score = 100 * (state.goals[team] - state.goals[opponent])
    if state.possession == team:
        r = state.positions[team][state.carrier][0]
        score += (r if team == A else ROWS - 1 - r) * 2
        score += 3
    elif state.possession == opponent:
        r = state.positions[opponent][state.carrier][0]
        score -= (r if opponent == A else ROWS - 1 - r) * 2
        score -= 3
    return score

def evaluate_structure(state, team):
    opponent = 1 - team
    score = 100 * (state.goals[team] - state.goals[opponent])
    own = state.positions[team]
    other = state.positions[opponent]
    if state.possession == team:
        carrier = own[state.carrier]
        support = min((abs(carrier[0]-p[0]) + abs(carrier[1]-p[1])
                       for i, p in enumerate(own) if i != state.carrier), default=0)
        score += 8 - support
    else:
        carrier = other[state.carrier]
        pressure = min((max(abs(carrier[0]-p[0]), abs(carrier[1]-p[1]))
                        for p in own), default=5)
        score += 5 - pressure
    return score

def random_agent(state, rng=random):
    actions = legal_actions(state)
    return rng.choice(actions) if actions else None

def greedy_agent(state, evaluator=evaluate_field):
    actions = legal_actions(state)
    if not actions:
        return None
    scored = [(evaluator(apply_action(state, a), state.turn), a) for a in actions]
    return max(scored, key=lambda x: x[0])[1]

## Minimax and alpha-beta search

In [ ]:
def minimax(state, depth, root_team, evaluator=evaluate_field, counters=None):
    if counters is not None:
        counters["nodes"] += 1
    if depth == 0 or terminal(state):
        return evaluator(state, root_team), None
    actions = legal_actions(state)
    if not actions:
        return evaluator(state, root_team), None
    maximizing = state.turn == root_team
    best_value = -math.inf if maximizing else math.inf
    best_action = None
    for action in actions:
        value, _ = minimax(apply_action(state, action), depth-1, root_team, evaluator, counters)
        if (maximizing and value > best_value) or (not maximizing and value < best_value):
            best_value, best_action = value, action
    return best_value, best_action

def alphabeta(state, depth, root_team, evaluator=evaluate_field, counters=None,
              alpha=-math.inf, beta=math.inf):
    if counters is not None:
        counters["nodes"] += 1
    if depth == 0 or terminal(state):
        return evaluator(state, root_team), None
    actions = legal_actions(state)
    if not actions:
        return evaluator(state, root_team), None
    maximizing = state.turn == root_team
    best_action = None
    if maximizing:
        value = -math.inf
        for action in actions:
            child_value, _ = alphabeta(apply_action(state, action), depth-1, root_team,
                                       evaluator, counters, alpha, beta)
            if child_value > value:
                value, best_action = child_value, action
            alpha = max(alpha, value)
            if alpha >= beta:
                break
        return value, best_action
    value = math.inf
    for action in actions:
        child_value, _ = alphabeta(apply_action(state, action), depth-1, root_team,
                                   evaluator, counters, alpha, beta)
        if child_value < value:
            value, best_action = child_value, action
        beta = min(beta, value)
        if alpha >= beta:
            break
    return value, best_action

def search_agent(state, depth=2, evaluator=evaluate_field, pruning=True):
    counters = {"nodes": 0}
    if pruning:
        value, action = alphabeta(state, depth, state.turn, evaluator, counters)
    else:
        value, action = minimax(state, depth, state.turn, evaluator, counters)
    return action, value, counters["nodes"]

## Monte Carlo tree search with random rollouts

In [ ]:
def rollout(state, root_team, max_steps=16, rng=random):
    current = state
    for _ in range(max_steps):
        if terminal(current):
            break
        actions = legal_actions(current)
        if not actions:
            break
        current = apply_action(current, rng.choice(actions))
    return evaluate_field(current, root_team)

def mcts_agent(state, simulations=80, rollout_steps=16, rng=random):
    actions = legal_actions(state)
    if not actions:
        return None
    totals = defaultdict(float)
    visits = defaultdict(int)
    root_team = state.turn
    for _ in range(simulations):
        action = rng.choice(actions)
        result = rollout(apply_action(state, action), root_team, rollout_steps, rng)
        totals[action] += result
        visits[action] += 1
    return max((a for a in actions if visits[a]), key=lambda a: totals[a] / visits[a])

## Play games and run tournaments

In [ ]:
def play_game(agent_a, agent_b, turn_budget=60, seed=0):
    rng = random.Random(seed)
    state = initial_state(turn_budget)
    agents = {A: agent_a, B: agent_b}
    while not terminal(state):
        agent = agents[state.turn]
        if agent == random_agent:
            action = agent(state, rng)
        else:
            action = agent(state)
        actions = legal_actions(state)
        if action not in actions:
            action = rng.choice(actions)
        state = apply_action(state, action)
    return state.goals

def tournament(agent_a, agent_b, games=20):
    outcomes = {"A_wins": 0, "B_wins": 0, "draws": 0}
    for i in range(games):
        if i % 2 == 0:
            score = play_game(agent_a, agent_b, seed=i)
            first_is_a = True
        else:
            score = play_game(agent_b, agent_a, seed=i)
            first_is_a = False
        if score[0] == score[1]:
            outcomes["draws"] += 1
        elif (score[0] > score[1]) == first_is_a:
            outcomes["A_wins"] += 1
        else:
            outcomes["B_wins"] += 1
    return outcomes

def greedy_field(s):
    return greedy_agent(s, evaluate_field)

def greedy_structure(s):
    return greedy_agent(s, evaluate_structure)

def alphabeta_field(s):
    return search_agent(s, depth=2, evaluator=evaluate_field, pruning=True)[0]

def alphabeta_structure(s):
    return search_agent(s, depth=2, evaluator=evaluate_structure, pruning=True)[0]

print("Greedy vs alpha-beta:", tournament(greedy_field, alphabeta_field, games=20))
print("Field vs structure:", tournament(alphabeta_field, alphabeta_structure, games=20))

## Pruning cost comparison and depth profile

In [ ]:
def search_profile(state, depths=(1, 2, 3), evaluator=evaluate_field):
    rows = []
    for depth in depths:
        for name, pruning in (("Minimax", False), ("Alpha-beta", True)):
            counters = {"nodes": 0}
            start = time.perf_counter()
            if pruning:
                value, action = alphabeta(state, depth, state.turn, evaluator, counters)
            else:
                value, action = minimax(state, depth, state.turn, evaluator, counters)
            elapsed = time.perf_counter() - start
            rows.append({"depth": depth, "algorithm": name, "nodes": counters["nodes"],
                         "seconds": elapsed, "value": value, "action": action})
    return rows

profile = search_profile(initial_state(30), depths=(1, 2, 3))
for row in profile:
    print(row)

## Additional experiments to complete the assignment's comparison matrix

The functions below let you repeat pruning measurements on a second reachable position and compare MCTS
with the deterministic baselines. The tournament helper supports alternating sides; use at least 20 games
per pairing before interpreting win rates. The quick demonstration above uses fewer games to keep notebook
execution time reasonable.

In [ ]:
def reachable_test_position(seed=7, steps=7, turn_budget=80):
    rng = random.Random(seed)
    state = initial_state(turn_budget)
    for _ in range(steps):
        actions = legal_actions(state)
        if not actions:
            break
        state = apply_action(state, rng.choice(actions))
    return state

second_position = reachable_test_position()
print("Second-position search profile:")
for row in search_profile(second_position, depths=(1, 2, 3)):
    print(row)

def mcts_agent_small(state):
    return mcts_agent(state, simulations=25, rollout_steps=10)

print("Greedy vs MCTS:", tournament(greedy_field, mcts_agent_small, games=20))
print("Alpha-beta vs MCTS:", tournament(alphabeta_field, mcts_agent_small, games=20))

## Important interpretation

- Alpha-beta pruning returns the same minimax value as plain minimax when both use the same depth and evaluation.
  Its benefit is fewer explored nodes; the amount depends heavily on move ordering.
- The field heuristic rewards territorial progress and possession. The structure heuristic rewards support
  distance or defensive pressure. Neither is a perfect model of tactical football.
- Random rollouts are a basic MCTS baseline. A stronger MCTS would maintain a search tree, use UCB selection,
  expand nodes, back-propagate values, and use carefully designed rollout policies.
- The small tournaments are demonstrations, not statistically strong claims. Increase games and use paired
  seeds before drawing conclusions.
- Search depth grows the tree rapidly. The branching factor changes by phase: possession turns permit dribbles,
  passes and shots; defensive turns permit repositioning moves.